In [1]:
from datasets import load_dataset, load_from_disk
import polars as pl
from sentence_transformers import SentenceTransformer
import faiss
from scipy.sparse import coo_array
import scipy
import os
import pandas as pd
import json
import pickle
import numpy as np
import datasets
import os
from datetime import datetime
import xxhash

NUM_PROC = 32
CACHE_DIR = "/home/jupyter/filestore/storage/"
datasets.config.IN_MEMORY_MAX_SIZE = 1073741824 * 16

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [2]:
DATA_PATH = "/home/jupyter/filestore/storage/datasets/user_events_20230501"

dataset = load_from_disk(DATA_PATH)

In [3]:
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(-1)

polars.config.Config

In [4]:
polars_ds = dataset.to_polars()

polars_ds = polars_ds.with_columns(
    pl.col("brand_name").fill_null(""),
    pl.col("item_condition_name").fill_null(""),
    pl.col("size_name").fill_null(""),
    pl.col("color").fill_null("")
)

In [5]:
unique_items = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= pd.to_datetime("2023-05-21"))
    .select("item_id", "name", "brand_name", "item_condition_name", "size_name", "color")
    .unique()
)

item_names = unique_items["name"].to_list()
item_ids = unique_items["item_id"].to_list()
brand_names = unique_items["brand_name"].to_list()
condition_names = unique_items["item_condition_name"].to_list()
size_names = unique_items["size_name"].to_list()
color_names = unique_items["color"].to_list()

In [6]:
item_info = [
    (item_names[i] + " " + brand_names[i] + " " + condition_names[i] + " " + size_names[i] + " " + color_names[i]).strip() for
    i in range(len(item_names))
]

In [16]:
item_info[:5]

['30 Ruby Red Glass Crystal Prism Lamp Chandelier Part Suncatcher dark pins arts  Like new',
 'Vintage Jurassic Park VELOCIRAPTOR JP18 1997 Snap Jaw Sound Mattel Good',
 'Barbie Doll and Barbie clothes Lot 2005 to 2015 Dolls Barbie Fair',
 'Home depot apron for dolls ♥️ Home Depot New',
 'Blue Light Filter Reading Glasses +1.00 NWT  New']

In [7]:
with open("data/complex_item_ids", "wb") as fp:
    pickle.dump(item_ids, fp)

In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 668.43it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
embeddings = model.encode(item_info, batch_size=4096, show_progress_bar=True, normalize_embeddings=True)

Batches: 100%|██████████| 1494/1494 [09:47<00:00,  2.54it/s]


In [ ]:
dim = 384
n_clusters = 2500

quantizer = faiss.IndexFlatIP(dim)
index = faiss.IndexIVFFlat(quantizer, dim, n_clusters)
index.train(embeddings)
index.add(embeddings)
index.nprobe = 64

In [11]:
m = 48
nbits = 8

quantizer = faiss.IndexFlatIP(dim)

ivfpq = faiss.IndexIVFPQ(
    quantizer,
    dim,
    n_clusters,
    m,
    nbits,
    faiss.METRIC_INNER_PRODUCT
)

opq = faiss.OPQMatrix(dim, m)
index = faiss.IndexPreTransform(opq, ivfpq)

index.train(embeddings)

index.add(embeddings)

In [12]:
ivf = faiss.extract_index_ivf(index)
ivf.nprobe = 64

In [13]:
ivfpq.use_precomputed_table = True

In [14]:
faiss.write_index(index, "data/complex_item_index_tiny.faiss")